# 02 — Project next IDX session with the TF15 model

Uses actual TF15 context through the newest completed candle. By default it
screens every stock with enough context and ranks the predicted first
15-minute candle of the next weekday. There is no liquidity pre-filter.
CPU is supported, but a full-universe run is substantially slower than CUDA.

Run notebook 01 first. If tomorrow is an IDX holiday, set `TARGET_DATE` manually.

In [ ]:
from pathlib import Path
import importlib.util

HERE = Path.cwd().resolve()
if HERE.name != "Daily Screener":
    HERE = HERE / "Daily Screener"
module_path = HERE / "project_tf15_next_session.py"
assert module_path.exists(), f"Open this notebook from the ISTL repository: {HERE}"
spec = importlib.util.spec_from_file_location("tf15_projection", module_path)
tf15 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(tf15)

## Configuration

In [ ]:
LOOKBACK_BARS = 240
SAMPLE_PATHS = 5       # use 1 for a quick CPU smoke test
BATCH_SIZE = None      # automatic: CPU=2, CUDA=16
TARGET_DATE = None     # example: "2026-08-04"; None = next weekday after latest actual bar

In [ ]:
ranking, forecast_paths, metadata = tf15.run_projection(
    lookback=LOOKBACK_BARS,
    paths=SAMPLE_PATHS,
    batch_size=BATCH_SIZE,
    target_date=TARGET_DATE,
)
metadata

In [ ]:
from IPython.display import display
top30 = ranking.head(30)
styled = top30.style.format({
    "anchor_close": "{:,.2f}", "expected_opening_bar_close": "{:,.2f}",
    "expected_return": "{:+.2%}", "median_return": "{:+.2%}",
    "probability_up": "{:.0%}", "downside_p10": "{:+.2%}",
}).background_gradient(subset=["expected_return"], cmap="RdYlGn")
display(styled)